# AutoCompose — GPT-2 Poem Fine-tuning (Fixed & Production-Grade)

**All issues from the initial experiment have been corrected:**
- BOS token properly added to the vocabulary via `add_special_tokens`
- `DataCollatorForLanguageModeling(mlm=False)` replaces the wrong `DataCollatorWithPadding`
- `model.eval()` and `total_val_loss = 0` moved outside the step loop  
- LR scheduler `num_training_steps` corrected for gradient accumulation (was 16× off)
- Loss logging uses unscaled loss (true per-batch loss is now reported correctly)
- Weight decay applied only to weight matrices, not biases or LayerNorm params
- Versioned checkpoints + best-model tracking by validation loss
- `model.generate()` replaces the manual greedy loop (KV-cache, sampling, BOS prepend)
- Full reproducibility seeding, `num_workers`, `pin_memory`, `GradScaler("cuda")`


## 1. Config

All hyperparameters live in one dictionary. Change here, nowhere else.

In [ ]:
# ──────────────────────────────────────────────────────────
# All tuneable knobs in one place.
# ──────────────────────────────────────────────────────────
CONFIG = {
    # Paths
    "data_dir":         "data",
    "data_file":        "anticipation.json",
    "checkpoint_dir":   "checkpoint",

    # Model
    "model_name":       "gpt2",

    # Training
    "epochs":                      3,
    "batch_size":                  4,
    "gradient_accumulation_steps": 16,      # effective batch = 4 × 16 = 64
    "learning_rate":               5e-5,
    "adam_epsilon":                1e-8,
    "weight_decay":                0.01,    # applied only to weight matrices
    "warmup_ratio":                0.06,    # 6 % of optimizer steps → warmup
    "max_grad_norm":               1.0,

    # Data
    "max_len":      256,
    "train_ratio":  0.95,
    "num_workers":  2,              # >0 overlaps CPU data-prep with GPU compute

    # Logging / Saving
    "log_every_n_steps":  100,      # log every N *dataloader* steps
    "save_every_n_steps": 1000,     # versioned checkpoint every N dataloader steps

    # Reproducibility
    "seed": 64,
}


## 2. Imports

In [ ]:
import os, json, time, random
from datetime import timedelta
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, random_split, RandomSampler, SequentialSampler

from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    DataCollatorForLanguageModeling,   # FIX: replaces DataCollatorWithPadding
    get_linear_schedule_with_warmup,
)


## 3. Reproducibility & Device

Setting every relevant seed in one function. `cudnn.deterministic = True` forces
CUDA ops to use deterministic algorithms; `benchmark = False` disables the
auto-tuner that picks different (potentially non-deterministic) kernels per run.

In [ ]:
def set_seed(seed: int) -> None:
    """Seed every RNG that PyTorch + Transformers touches."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU        : {props.name}")
    print(f"  Total VRAM : {props.total_memory / 1024**3:.2f} GB")
    print(f"  Allocated  : {torch.cuda.memory_allocated() / 1024**3:.3f} GB")


## 4. Tokenizer & Model

### Fix: BOS token added via `add_special_tokens`

The original code did `tokenizer.bos_token = "<|startoftext|>"`, which only set an
attribute — it did **not** insert the token into the vocabulary. The string
`"<|startoftext|>"` was therefore tokenised into ~7 subword fragments every time it
appeared in training data.

The fix is `tokenizer.add_special_tokens({"bos_token": ...})`, which:
1. Adds the token string as a single new entry in the vocabulary
2. Increments `len(tokenizer)` from 50 257 → 50 258
3. Allows `model.resize_token_embeddings(len(tokenizer))` to actually grow the
   embedding matrix by one row (randomly initialised, learned during fine-tuning)


In [ ]:
BOS = "<|startoftext|>"

tokenizer = GPT2Tokenizer.from_pretrained(CONFIG["model_name"])

# ── FIX: add BOS as a genuine vocabulary entry, not just an attribute ──────
tokenizer.add_special_tokens({"bos_token": BOS})

# GPT-2 has no dedicated PAD token; EOS doubles as PAD (right-padding).
# The collator masks padding positions with -100 in labels, so the loss
# ignores them regardless of what the pad token ID is.
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(CONFIG["model_name"])

# Grow the embedding matrix to accommodate the new BOS token row.
# Without this, any input with the new BOS token ID would be out-of-bounds.
model.resize_token_embeddings(len(tokenizer))

model = model.to(device)

print(f"Tokenizer vocab size : {len(tokenizer)}")
print(f"Model embedding size : {model.get_input_embeddings().num_embeddings}")
print(f"BOS : '{tokenizer.bos_token}' (id {tokenizer.bos_token_id})")
print(f"EOS : '{tokenizer.eos_token}' (id {tokenizer.eos_token_id})")
print(f"PAD : '{tokenizer.pad_token}' (id {tokenizer.pad_token_id})")
assert len(tokenizer) == model.get_input_embeddings().num_embeddings,     "Tokenizer and model vocab sizes are out of sync!"


## 5. Dataset

`PoemDataset` pre-tokenises the entire corpus at construction time — correct
for a small dataset because it removes per-epoch tokenisation overhead.
`padding=False` defers padding to the collator, so each batch pads only to its
own longest sequence rather than globally to `max_len`.

`tokenizer.bos_token` is used symbolically so the dataset doesn't hard-code a
string literal that could drift from the tokenizer state.


In [ ]:
class PoemDataset(Dataset):
    """
    Pre-tokenises poems once. Each sample is a dict with:
      input_ids       : LongTensor [seq_len]  (variable length, no padding)
      attention_mask  : LongTensor [seq_len]  (all 1s — padding added by collator)
    """

    def __init__(self, poems: list, tokenizer: GPT2Tokenizer, max_len: int = 256):
        self.samples = []

        for poem in poems:
            # Use tokenizer.bos_token so this never drifts from the tokenizer state
            text = tokenizer.bos_token + poem + tokenizer.eos_token

            enc = tokenizer(
                text,
                max_length=max_len,
                truncation=True,
                padding=False,          # collator pads batch-by-batch
                return_attention_mask=True,
            )
            self.samples.append({
                "input_ids":      torch.tensor(enc["input_ids"],      dtype=torch.long),
                "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            })

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        return self.samples[idx]


## 6. Data Loading

### Fix: `DataCollatorForLanguageModeling(mlm=False)`

`DataCollatorWithPadding` is a classification collator — it only pads `input_ids`
and `attention_mask`. It does not produce a `labels` key.

`DataCollatorForLanguageModeling` with `mlm=False` is the correct CLM collator:
- Pads each batch to its longest sequence (+ `pad_to_multiple_of=8` for Tensor Core alignment)
- Produces a `labels` key that is a copy of `input_ids` with padding positions
  already set to `-100` (ignored by `CrossEntropyLoss` internally)
- Removes the manual `b_labels[b_masks == 0] = -100` dance from the training loop


In [ ]:
data_path = Path(CONFIG["data_dir"]) / CONFIG["data_file"]
with open(data_path, "r") as f:
    raw_data = json.load(f)

poems_text = [entry["poem"] for entry in raw_data]
print(f"Total poems loaded: {len(poems_text)}")

full_dataset = PoemDataset(
    poems=poems_text,
    tokenizer=tokenizer,
    max_len=CONFIG["max_len"],
)

# ── Length analysis ────────────────────────────────────────────────────────
lengths = np.array([s["attention_mask"].sum().item() for s in full_dataset])
n_truncated = int((lengths == CONFIG["max_len"]).sum())

print(f"\nToken lengths — min: {lengths.min()}, max: {lengths.max()}, "
      f"mean: {lengths.mean():.1f}")
for pct in [50, 90, 95, 99]:
    print(f"  P{pct}: {np.percentile(lengths, pct):.0f}")
print(f"Poems truncated at max_len={CONFIG['max_len']}: "
      f"{n_truncated} ({100*n_truncated/len(lengths):.1f}%)")

# ── Train / Val split ──────────────────────────────────────────────────────
train_size = int(CONFIG["train_ratio"] * len(full_dataset))
val_size   = len(full_dataset) - train_size

# Seed the generator so the split is reproducible across runs
train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(CONFIG["seed"]),
)
print(f"\nTrain: {train_size} | Val: {val_size}")

# ── Collator ───────────────────────────────────────────────────────────────
# FIX: DataCollatorForLanguageModeling(mlm=False) is the correct CLM collator.
# It produces labels with padding masked to -100 automatically.
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,   # aligns tensors to 8-token boundaries for Tensor Cores
)

# ── DataLoaders ────────────────────────────────────────────────────────────
_pin = device.type == "cuda"   # pin_memory only makes sense on GPU

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    sampler=RandomSampler(train_dataset),
    collate_fn=data_collator,
    num_workers=CONFIG["num_workers"],  # FIX: >0 overlaps CPU prep with GPU
    pin_memory=_pin,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    sampler=SequentialSampler(val_dataset),
    collate_fn=data_collator,
    num_workers=CONFIG["num_workers"],
    pin_memory=_pin,
)

print(f"\nTrain loader: {len(train_loader)} batches/epoch")
print(f"Val   loader: {len(val_loader)} batches")

# Quick sanity-check: confirm the collator now produces labels
sample_batch = next(iter(train_loader))
print(f"\nBatch keys    : {list(sample_batch.keys())}")
print(f"input_ids shape: {sample_batch['input_ids'].shape}")
print(f"labels shape   : {sample_batch['labels'].shape}")
print(f"Unique label values (should include -100): "
      f"{sample_batch['labels'].unique().tolist()[:10]}")


## 7. Optimizer, Scheduler & Scaler

### Fix: `num_training_steps` corrected for gradient accumulation

The original code set `num_training_steps = len(train_loader) * epochs`.
But `scheduler.step()` is only called at optimizer steps (every
`gradient_accumulation_steps` dataloader iterations). With `gradient_accumulation_steps=16`,
only 1/16th of `num_training_steps` scheduler calls ever happen, so the LR
barely decayed from its peak — you were essentially training with a flat LR.

The correct formula: `total_optimizer_steps = (len(train_loader) // grad_accum) * epochs`.

### Weight decay parameter grouping

`AdamW`'s weight decay is a form of L2 regularisation. Applying it to biases and
`LayerNorm` parameters is incorrect — those parameters should not be shrunk toward
zero. The fix: split `model.parameters()` into two groups.


In [ ]:
# ── Optimizer: weight decay only on weight matrices ────────────────────────
_no_decay = {"bias", "LayerNorm.weight"}

param_groups = [
    {
        "params": [p for n, p in model.named_parameters()
                   if not any(nd in n for nd in _no_decay)],
        "weight_decay": CONFIG["weight_decay"],
    },
    {
        "params": [p for n, p in model.named_parameters()
                   if any(nd in n for nd in _no_decay)],
        "weight_decay": 0.0,
    },
]

optimizer = AdamW(
    param_groups,
    lr=CONFIG["learning_rate"],
    eps=CONFIG["adam_epsilon"],
)

# ── Scheduler: FIX num_training_steps ─────────────────────────────────────
# scheduler.step() is called once per optimizer step, NOT per dataloader step.
# num_training_steps must equal the number of OPTIMIZER steps, or the LR
# schedule will be miscalibrated by a factor of gradient_accumulation_steps.
total_dataloader_steps = len(train_loader) * CONFIG["epochs"]
total_optimizer_steps  = (len(train_loader) // CONFIG["gradient_accumulation_steps"])                           * CONFIG["epochs"]
warmup_steps = max(1, int(total_optimizer_steps * CONFIG["warmup_ratio"]))

print(f"Total dataloader steps : {total_dataloader_steps}")
print(f"Total optimizer steps  : {total_optimizer_steps}  "
      f"(= dataloader_steps / grad_accum)")
print(f"Warmup steps           : {warmup_steps}  "
      f"({CONFIG['warmup_ratio']*100:.0f}% of optimizer steps)")

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps,
)

# ── GradScaler: FIX — pass device type explicitly ─────────────────────────
# torch.amp.GradScaler() without an argument is deprecated in PyTorch >= 2.0.
scaler = torch.amp.GradScaler("cuda")


## 8. Utilities

In [ ]:
def format_time(elapsed: float) -> str:
    return str(timedelta(seconds=int(round(elapsed))))

def log_gpu_memory() -> None:
    if device.type == "cuda":
        alloc    = torch.cuda.memory_allocated()  / 1024**3
        reserved = torch.cuda.memory_reserved()   / 1024**3
        print(f"  GPU — allocated: {alloc:.3f} GB | reserved: {reserved:.3f} GB")


## 9. Overfitting Sanity Check (optional, run before full training)

Before committing to a full training run, verify the loop is correct by
intentionally driving loss to near-zero on a single batch. If the model
cannot overfit one batch, there is a bug in the forward pass, label
construction, or optimizer — and full training would silently fail.

The original overfitting test block had a variable ordering bug:
`b_labels[b_masks == 0] = -100` ran *before* `b_labels = batch["input_ids"].to(device)`,
so the masking was immediately overwritten. This cell has the correct order.

*Uncomment the call at the bottom to run before training.*


In [ ]:
def overfit_sanity_check(
    model: GPT2LMHeadModel,
    loader: DataLoader,
    device: torch.device,
    steps: int = 100,
) -> None:
    """
    Force the model to memorise a single batch.
    Expected outcome: loss drops to < 0.5 within ~100 steps.
    If it does not, something is wrong with the training loop.
    """
    model.train()
    _opt = AdamW(model.parameters(), lr=1e-4)

    # Grab one batch and keep it fixed for all steps
    batch = next(iter(loader))
    ids    = batch["input_ids"].to(device)
    masks  = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)  # collator already masks padding → -100

    print("── Overfitting sanity check ───────────────────────────────────")
    initial_loss = None
    for step in range(steps):
        _opt.zero_grad()
        outputs = model(input_ids=ids, attention_mask=masks, labels=labels)
        loss = outputs.loss
        loss.backward()
        _opt.step()
        if step == 0:
            initial_loss = loss.item()
        if (step + 1) % 20 == 0:
            print(f"  step {step+1:3d}: loss = {loss.item():.4f}")

    final_loss = loss.item()
    reduction  = (initial_loss - final_loss) / initial_loss * 100
    status = "✓ PASSED" if final_loss < 0.5 else "✗ WARNING: loss did not converge"
    print(f"  Initial: {initial_loss:.4f}  →  Final: {final_loss:.4f}  "
          f"({reduction:.1f}% reduction)")
    print(f"  {status}")
    print("───────────────────────────────────────────────────────────────")

    # Restore model to pre-check state (re-load from disk avoids any weight drift)
    print("  Restoring model weights from pretrained checkpoint...")
    global model
    model = GPT2LMHeadModel.from_pretrained(CONFIG["model_name"])
    model.resize_token_embeddings(len(tokenizer))
    model = model.to(device)

# Uncomment to run before training:
# overfit_sanity_check(model, train_loader, device, steps=100)


## 10. Training Loop

### Critical fixes in this cell

| Bug | Location in original | Fix |
|---|---|---|
| `model.eval()` inside step loop | 8-space indent inside `for step` | Moved to after the step loop |
| `total_val_loss = 0` inside step loop | 8-space indent inside `for step` | Moved to after the step loop |
| Loss logging used pre-divided loss | `batch_loss = loss.item()` after `loss /= accum` | Multiply back: `outputs.loss.item()` |
| No avg train loss logged | — | `avg_train_loss` computed and printed per epoch |
| No best-model tracking | Overwrote same dir every checkpoint | `best_val_loss` tracked; best model saved separately |
| Checkpoints overwrite each other | Single path for all checkpoints | Versioned paths: `epoch{e}_step{s}` |
| `outputs[0]` in validation | Positional indexing | `outputs.loss` — named attribute, robust to API changes |
| `total_val_loss` held graph tensor | `+= outputs[0]` | `+= outputs.loss.item()` |


In [ ]:
def train(
    model:       GPT2LMHeadModel,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    optimizer:    AdamW,
    scheduler,
    scaler:       torch.amp.GradScaler,
    config:       dict,
    device:       torch.device,
) -> list:
    """
    Full training loop with all bugs fixed.
    Returns a list of per-epoch dicts with train_loss, val_loss, epoch_time.
    """
    ckpt_dir = Path(config["checkpoint_dir"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    best_val_loss    = float("inf")
    training_history = []
    global_step      = 0     # counts optimizer (gradient) steps
    accum_steps      = config["gradient_accumulation_steps"]
    total_train_start = time.perf_counter()

    for epoch in range(config["epochs"]):
        epoch_start = time.perf_counter()

        # ╔══════════════════════════════════════════════════════════════╗
        # ║  Training phase                                              ║
        # ╚══════════════════════════════════════════════════════════════╝
        model.train()
        optimizer.zero_grad()

        total_train_loss  = 0.0
        num_train_batches = 0

        for step, batch in enumerate(train_loader):

            input_ids  = batch["input_ids"].to(device)
            attn_mask  = batch["attention_mask"].to(device)
            labels     = batch["labels"].to(device)   # -100 at padding positions

            # FP16 forward pass
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attn_mask,
                    labels=labels,
                )
                # Divide loss BEFORE backward so accumulated gradients stay
                # in the correct scale relative to a single optimizer step
                loss = outputs.loss / accum_steps

            scaler.scale(loss).backward()

            # FIX: log the TRUE per-batch loss, not the divided value
            # outputs.loss.item() is the unscaled loss for this batch
            true_batch_loss   = outputs.loss.item()
            total_train_loss += true_batch_loss
            num_train_batches += 1

            # ── Periodic step logging ─────────────────────────────────
            if (step + 1) % config["log_every_n_steps"] == 0:
                current_lr = scheduler.get_last_lr()[0]
                elapsed    = time.perf_counter() - epoch_start
                print(
                    f"  Epoch {epoch+1}/{config['epochs']} | "
                    f"Step {step+1}/{len(train_loader)} | "
                    f"Loss: {true_batch_loss:.4f} | "
                    f"LR: {current_lr:.2e} | "
                    f"Elapsed: {format_time(elapsed)}"
                )

            # ── Optimizer step at accumulation boundary ───────────────
            is_accum_step = (step + 1) % accum_steps == 0
            is_last_step  = (step + 1) == len(train_loader)

            if is_accum_step or is_last_step:
                # unscale_ MUST come before clip_grad_norm_ when using GradScaler;
                # the scaler inflates gradients internally, so clipping on
                # raw scaled gradients would be meaningless
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=config["max_grad_norm"],
                    foreach=True,
                )
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            # ── Versioned mid-epoch checkpoint ────────────────────────
            if (step + 1) % config["save_every_n_steps"] == 0:
                ckpt_path = ckpt_dir / f"epoch{epoch+1}_step{step+1}"
                ckpt_path.mkdir(exist_ok=True)
                model.save_pretrained(ckpt_path)
                tokenizer.save_pretrained(ckpt_path)
                print(f"  ✓ Checkpoint: {ckpt_path}")

        avg_train_loss = total_train_loss / num_train_batches

        # ╔══════════════════════════════════════════════════════════════╗
        # ║  Validation phase                                            ║
        # ║  FIX: model.eval() and total_val_loss = 0 are HERE,         ║
        # ║       OUTSIDE the step loop — not inside it                 ║
        # ╚══════════════════════════════════════════════════════════════╝
        model.eval()
        total_val_loss = 0.0   # FIX: reset once per epoch, not once per step

        with torch.no_grad():
            for val_batch in val_loader:
                v_ids    = val_batch["input_ids"].to(device)
                v_masks  = val_batch["attention_mask"].to(device)
                v_labels = val_batch["labels"].to(device)

                with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                    val_out = model(
                        input_ids=v_ids,
                        attention_mask=v_masks,
                        labels=v_labels,
                    )

                # FIX: .item() detaches the scalar from the autograd graph
                # to avoid holding onto the computation graph across batches
                total_val_loss += val_out.loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        epoch_time   = time.perf_counter() - epoch_start

        print(f"\n{'═'*62}")
        print(f"  Epoch {epoch+1}/{config['epochs']} Complete")
        print(f"  Avg Train Loss : {avg_train_loss:.4f}")
        print(f"  Avg Val Loss   : {avg_val_loss:.4f}")
        print(f"  Epoch Time     : {format_time(epoch_time)}")
        log_gpu_memory()
        print(f"{'═'*62}\n")

        # ── Best model tracking ───────────────────────────────────────
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_path = ckpt_dir / "best_model"
            best_path.mkdir(exist_ok=True)
            model.save_pretrained(best_path)
            tokenizer.save_pretrained(best_path)
            print(f"  ★  New best model saved  (val_loss = {best_val_loss:.4f})")

        training_history.append({
            "epoch":      epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss":   avg_val_loss,
            "epoch_time": epoch_time,
        })

        # Return to train mode for the next epoch
        model.train()

    total_time = time.perf_counter() - total_train_start
    print(f"Training complete in {format_time(total_time)}")
    print(f"Best validation loss: {best_val_loss:.4f}")
    return training_history


## 11. Run Training

In [ ]:
training_history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    config=CONFIG,
    device=device,
)


## 12. Loss Curves

In [ ]:
epochs_axis  = [h["epoch"]      for h in training_history]
train_losses = [h["train_loss"] for h in training_history]
val_losses   = [h["val_loss"]   for h in training_history]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_axis, train_losses, marker="o", linewidth=2, label="Train loss")
ax.plot(epochs_axis, val_losses,   marker="s", linewidth=2, label="Val loss",   linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("AutoCompose — Training vs Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("loss_curve.png", dpi=150)
plt.show()
print("Loss curve saved to loss_curve.png")


## 13. Final Model Save

In [ ]:
final_path = Path(CONFIG["checkpoint_dir"]) / "final_model"
final_path.mkdir(exist_ok=True)
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f"Final model saved to: {final_path}")


## 14. Inference — `generate_poem()`

### Fixes vs. the original `generate()` function

| Issue | Original | Fixed |
|---|---|---|
| No BOS prefix | Prompt sent raw | `tokenizer.bos_token + prompt` prepended |
| Greedy decoding | `argmax` loop | `model.generate()` with `do_sample=True`, `top_p`, `temperature` |
| Hardcoded EOS ID | `end_of_text_id = 50256` | `tokenizer.eos_token_id` |
| Manual autoregressive loop | O(N²) attention, no KV-cache | `model.generate()` uses KV-cache by default |
| No repetition control | Repeats phrases | `repetition_penalty=1.3` |

The model prepended `tokenizer.bos_token` to every poem during training.
At inference time, the model therefore expects the BOS token as the very
first token. Omitting it creates a distribution mismatch that degrades quality.


In [ ]:
def generate_poem(
    prompt:            str,
    model:             GPT2LMHeadModel,
    tokenizer:         GPT2Tokenizer,
    device:            torch.device,
    max_new_tokens:    int   = 200,
    temperature:       float = 0.9,
    top_k:             int   = 50,
    top_p:             float = 0.92,
    repetition_penalty: float = 1.3,
    num_return_sequences: int = 1,
) -> list:
    """
    Generate poems with nucleus sampling via model.generate().

    - BOS token is prepended to match training-time distribution.
    - KV-cache is used internally by model.generate() for O(N) generation.
    - Returns a list of decoded strings (length = num_return_sequences).
    """
    model.eval()

    # FIX: prepend BOS to match training distribution
    input_text = tokenizer.bos_token + prompt
    input_ids  = tokenizer.encode(input_text, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            eos_token_id=tokenizer.eos_token_id,       # FIX: use tokenizer attribute
            pad_token_id=tokenizer.pad_token_id,
            num_return_sequences=num_return_sequences,
        )

    return [
        tokenizer.decode(seq, skip_special_tokens=True)
        for seq in output_ids
    ]


In [ ]:
generated = generate_poem(
    prompt="What a mystery",
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_new_tokens=150,
    temperature=0.9,
    top_p=0.92,
    num_return_sequences=3,
)

for i, poem in enumerate(generated, 1):
    print(f"─── Generated poem {i} ─────────────────────────────────")
    print(poem)
    print()


## 15. Load Best Model for Inference

In [ ]:
best_model_path = Path(CONFIG["checkpoint_dir"]) / "best_model"

best_model     = GPT2LMHeadModel.from_pretrained(best_model_path).to(device)
best_tokenizer = GPT2Tokenizer.from_pretrained(best_model_path)

print(f"Loaded best model from: {best_model_path}")
print(f"Best tokenizer vocab size: {len(best_tokenizer)}")

best_poems = generate_poem(
    prompt="In the silence of",
    model=best_model,
    tokenizer=best_tokenizer,
    device=device,
    max_new_tokens=150,
    temperature=0.85,
    top_p=0.90,
    num_return_sequences=2,
)

for i, poem in enumerate(best_poems, 1):
    print(f"─── Best model — poem {i} ──────────────────────────────")
    print(poem)
    print()
